# Comprehensive ECG Data Analysis & Preparation
### By: Advanced AI Data Scientist

Ek Data Scientist ka kaam sirf code likhna nahi, balkay **Data ko samajhna (Understand)** aur us se **Insights nikalna** hai. Is notebook mein hum PTB-XL ECG dataset ko medically aur technically deep analyze karenge.

**Key Objectives:**
1. **Medical Dictionary Exploration:** `scp_statements.csv` ko samajhna ke kis Superclass (Badi Beemari) ke andar konsi Subclass (Choti Beemari) aati hai.
2. **Data Cleansing & Transformation:** Fuzool data ko filter karna aur `{}` codes ko computer-readable (0 aur 1) format mein badalna.
3. **In-depth EDA:** Class imbalances, demographic distributions, aur beemariyon ke aapas mein talluq (co-occurrence) ko wazeh karna.

In [ ]:
# Required Libraries
!pip install pandas numpy matplotlib seaborn -q

import pandas as pd
import numpy as np
import ast
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries imported successfully!')

--- 
## 1. Data Loading & Understanding the Medical Labels
Sab se pehle hum data load karenge aur dekhenge ke PTB-XL ne beemariyon ko kis tarah darja bandi (categorize) kiya hai.

In [ ]:
df = pd.read_csv('ptbxl_database.csv')
scp = pd.read_csv('scp_statements.csv', index_col=0)

print(f"Total ECG Records: {df.shape[0]}")
print(f"Total Columns in Raw Data: {df.shape[1]}")

# Sirf Diagnostic (Beemari wali) statements filter karein
scp_diagnostic = scp[scp['diagnostic'] == 1.0]
print(f"\nTotal Diagnostic Statements (Unique Diseases): {scp_diagnostic.shape[0]}")

### Superclasses aur Subclasses ka Tree
AI model ke liye 71 choti beemariyon ko predict karna mushkil hai. Isliye hum unhein unki main **Superclasses** mein dekhte hain.

In [ ]:
print("---- SUPERCLASSES AUR UNKI SUBCLASSES ----\n")
grouped = scp_diagnostic.groupby('diagnostic_class')
for superclass, group in grouped:
    subclasses = group['diagnostic_subclass'].dropna().unique()
    print(f"🩺 SUPERCLASS: {superclass}")
    print(f"   Subclasses (Categories): {', '.join(subclasses)}")
    print(f"   Total Specific Codes in this class: {len(group)}\n")

**Data Scientist Insight:** 
Upar diye gaye nateeje se zahir hai ke `MI` (Heart Attack) ke andar mukhtalif types hain (jaise IMI, AMI). Hamara AI pehle general `MI` predict karega. Agar model successful hua, toh next step mein hum subclasses pe ja sakte hain.

--- 
## 2. Feature Selection & Data Cleaning
Humein model ke liye sirf demographics aur signal path chahiye.

In [ ]:
columns_to_keep = [
    'ecg_id', 'patient_id', 'age', 'sex', 'height', 'weight', 
    'pacemaker', 'strat_fold', 'scp_codes', 'filename_lr'
]
df_clean = df[columns_to_keep].copy()

# Imputation (Handling Missing Data logically)
df_clean['age'] = df_clean['age'].fillna(df_clean['age'].median())
df_clean['sex'] = df_clean['sex'].fillna(0)
df_clean['height'] = df_clean['height'].fillna(df_clean['height'].median())
df_clean['weight'] = df_clean['weight'].fillna(df_clean['weight'].median())
df_clean['pacemaker'] = df_clean['pacemaker'].notna().astype(int)

print("Data completely cleaned! No missing values remaining.")

--- 
## 3. Parsing Dictionary `{}` and Multi-label Encoding
Har ECG ki `scp_codes` dictionary ko read kar ke usmein se 100.0 score wali confirmed superclasses nikalenge.

In [ ]:
def extract_superclasses(codes_str):
    try:
        codes_dict = ast.literal_eval(codes_str)
    except:
        return []
    
    classes = set()
    for code, score in codes_dict.items():
        if score == 100.0 and code in scp_diagnostic.index:
            classes.add(scp_diagnostic.loc[code, 'diagnostic_class'])
    return list(classes)

df_clean['superclasses'] = df_clean['scp_codes'].apply(extract_superclasses)

# Create Binary Columns (0/1)
all_superclasses = ['NORM', 'MI', 'STTC', 'CD', 'HYP']
for sc in all_superclasses:
    df_clean[sc] = df_clean['superclasses'].apply(lambda x: 1 if sc in x else 0)

# Drop records with NO confirmed diagnostic class
df_clean['total_classes'] = df_clean[all_superclasses].sum(axis=1)
df_final = df_clean[df_clean['total_classes'] > 0].copy()

df_final.drop(columns=['total_classes', 'superclasses', 'scp_codes'], inplace=True)

print("Multi-label Encoding Successful!")
print(f"Total Confirmed ECG Records for Model Training: {df_final.shape[0]}")

### Class Instances Counts (Beemariyon ki Ginti)
Dekhte hain ke kis beemari ke kitne mareez hain.

In [ ]:
class_counts = df_final[all_superclasses].sum().sort_values(ascending=False)
print("Instances per Superclass:")
print("-" * 25)
for cls, count in class_counts.items():
    print(f"{cls:<5} : {count} cases")
print("-" * 25)
print("Insight: Is data mein severely Class Imbalance hai. NORM sab se zyada hai. Model train karte waqt class weights lazmi dena honge!")

--- 
## 4. In-depth Exploratory Data Analysis (EDA)
Ab hum graphs ke zariye un patterns ko dhoondenge jo aam ankh se nazar nahi aate.

In [ ]:
df_melted = df_final.melt(id_vars=['age', 'sex', 'weight'], value_vars=all_superclasses, var_name='Superclass', value_name='Presence')
df_present = df_melted[df_melted['Presence'] == 1].copy()
df_present['Sex_Label'] = df_present['sex'].map({0.0: 'Male', 1.0: 'Female'})

sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(2, 2, figsize=(18, 12))
fig.suptitle('Comprehensive EDA for ECG Superclasses', fontsize=20, fontweight='bold', y=1.02)

# 1. Age Distribution Boxplot
sns.boxplot(x='Superclass', y='age', data=df_present, palette='Set2', ax=axes[0, 0])
axes[0, 0].set_title('Age Distribution (Kaunsi umar mein konsi beemari?)', fontsize=13)
axes[0, 0].set_ylabel('Age')

# 2. Sex Distribution Countplot
sns.countplot(x='Superclass', hue='Sex_Label', data=df_present, palette='pastel', ax=axes[0, 1])
axes[0, 1].set_title('Gender Distribution (Mard vs Aurat Risk)', fontsize=13)
axes[0, 1].set_ylabel('Total Cases')

# 3. Weight Distribution Violin Plot
sns.violinplot(x='Superclass', y='weight', data=df_present, palette='muted', ax=axes[1, 0])
axes[1, 0].set_title('Weight Impact on Conditions', fontsize=13)
axes[1, 0].set_ylabel('Weight (kg)')

# 4. Co-occurrence Matrix (Heatmap)
co_matrix = df_final[all_superclasses].T.dot(df_final[all_superclasses])
for sc in all_superclasses:
    co_matrix.loc[sc, sc] = 0  # Ignore self-matching

sns.heatmap(co_matrix, annot=True, fmt='d', cmap='Reds', ax=axes[1, 1], cbar=True)
axes[1, 1].set_title('Disease Co-occurrence (Beemariyan Ek Sath)', fontsize=13)

plt.tight_layout()
plt.show()

--- 
## 5. Exporting Prepared Data
Ab hamara data modeling ke liye bilkul tayar hai.

In [ ]:
df_final.to_csv('ptbxl_cleaned_data_for_model.csv', index=False)
print("Data exported successfully as 'ptbxl_cleaned_data_for_model.csv'!")